In [ ]:
# ======================================================
# Lab: Regularization & Dropout in Neural Networks
# ======================================================
# In this lab, you will:
# 1. Compare training with and without L2 regularization
# 2. Add dropout and see its effect on generalization
# 3. Observe dropout randomness (train vs eval mode)
# 4. Visualize dropped neurons with heatmaps
# ======================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Data (MNIST)
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Split train into train/val
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_dataset, val_dataset = random_split(train_data, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_data, batch_size=64)

# 2. Define Model
class SimpleNN(nn.Module):
    def __init__(self, use_dropout=False, p=0.5):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(p=p) if use_dropout else nn.Identity()

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        x = self.drop(x)
        x = self.relu(self.fc2(x))
        x = self.drop(x)
        x = self.fc3(x)
        return x

# 3. Training & Evaluation Helpers
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X.size(0)
        _, predicted = outputs.max(1)
        correct += (predicted == y).sum().item()
        total += y.size(0)
    return running_loss/total, correct/total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            running_loss += loss.item() * X.size(0)
            _, predicted = outputs.max(1)
            correct += (predicted == y).sum().item()
            total += y.size(0)
    return running_loss/total, correct/total

# 4. Run Experiments
criterion = nn.CrossEntropyLoss()

def run_experiment(use_dropout=False, weight_decay=0, p=0.5):
    model = SimpleNN(use_dropout=use_dropout, p=p).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(5):  # keep short for demo
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        train_losses.append(tr_loss); val_losses.append(val_loss)
        train_accs.append(tr_acc); val_accs.append(val_acc)

        print(f"Epoch {epoch+1}: Train Acc={tr_acc:.3f}, Val Acc={val_acc:.3f}")

    return train_losses, val_losses, train_accs, val_accs

# Case A: No regularization
loss_A, val_loss_A, acc_A, val_acc_A = run_experiment(use_dropout=False, weight_decay=0)

# Case B: L2 regularization
loss_B, val_loss_B, acc_B, val_acc_B = run_experiment(use_dropout=False, weight_decay=1e-4)

# Case C: Dropout
loss_C, val_loss_C, acc_C, val_acc_C = run_experiment(use_dropout=True, weight_decay=0, p=0.5)

# 5. Visualize Training vs Validation
def plot_curves(train, val, title, ylabel):
    plt.plot(train, label="Train")
    plt.plot(val, label="Val")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.legend()
    plt.show()

print("\n--- Case A: No Regularization ---")
plot_curves(acc_A, val_acc_A, "Accuracy (No Regularization)", "Accuracy")

print("\n--- Case B: L2 Regularization ---")
plot_curves(acc_B, val_acc_B, "Accuracy (L2)", "Accuracy")

print("\n--- Case C: Dropout ---")
plot_curves(acc_C, val_acc_C, "Accuracy (Dropout)", "Accuracy")

# 6. Dropout Variance Demo
print("\n=== Dropout Variance Demo ===")
model = SimpleNN(use_dropout=True, p=0.5).to(device)
x = torch.randn(1, 28*28).to(device)

model.eval()  # dropout off
print("\nEval mode (dropout OFF):")
for i in range(3):
    out = model(x)
    print(f"Run {i+1}: {out[0,:5].cpu().detach().numpy()}")

model.train()  # dropout on
print("\nTrain mode (dropout ON):")
for i in range(3):
    out = model(x)
    print(f"Run {i+1}: {out[0,:5].cpu().detach().numpy()}")

# 7. Visualizing Dropped-Out Neurons
print("\n=== Dropout Visualization ===")
drop_layer = nn.Dropout(p=0.5)
x = torch.ones(20, 10)  # 20 samples, 10 neurons

drop_layer.train()
out = drop_layer(x).numpy()

plt.figure(figsize=(6,4))
sns.heatmap(out, annot=False, cbar=False, cmap="coolwarm")
plt.title("Dropout Visualization (0 = dropped neuron)")
plt.xlabel("Neuron index")
plt.ylabel("Sample index")
plt.show()
